In [38]:
import os

os.environ["PATH"] = "/Users/srijanshukla/.local/bin:" + os.environ["PATH"]

In [39]:
import os
from agents import set_tracing_export_api_key
from agents.extensions.models.litellm_model import LitellmModel

openai_api_key = os.getenv('OPENAI_API_KEY')
if not openai_api_key:
    print("WARNING: OPENAI_API_KEY not found. Tracing will be disabled.")
else:
    set_tracing_export_api_key(openai_api_key)
    print("OpenAI tracing configured")

os.environ['GEMINI_API_KEY'] = os.getenv('GOOGLE_API_KEY')

gemini_model = LitellmModel(
    model="gemini/gemini-2.0-flash",  
)

print("Gemini model with LitellmModel configured")

OpenAI tracing configured
Gemini model with LitellmModel configured


In [40]:
import os
from dotenv import load_dotenv
from agents import Agent, Runner, trace, Tool
from agents.mcp import MCPServerStdio
from IPython.display import Markdown, display
from datetime import datetime
from accounts_client import read_accounts_resource, read_strategy_resource
from accounts import Account

load_dotenv(override=True)

True

In [41]:
from pydantic import BaseModel

class ProposedTrade(BaseModel):
    action: str
    reasoning: str
    risks: str
    expected_outcome: str
    symbol: str
    quantity: int


In [42]:
# human_approval_file.py
import json
import asyncio
import pathlib
from agents import function_tool
from pydantic import BaseModel
from typing import Optional

class ProposedTrade(BaseModel):
    action: str
    reasoning: str
    risks: str
    expected_outcome: str
    symbol: str
    quantity: int


PENDING_FILE = pathlib.Path("./pending_trade.json")
POLL_INTERVAL = 0.5   # seconds
TIMEOUT = 300         # seconds (adjust as needed)


async def _wait_for_decision(timeout: int = TIMEOUT) -> str:
    """
    Wait for the pending file to contain:
        {"decision": "yes"} or {"decision": "no"}
    """
    start_time = asyncio.get_event_loop().time()

    while True:
        if PENDING_FILE.exists():
            try:
                contents = json.loads(PENDING_FILE.read_text(encoding="utf-8"))
                dec = contents.get("decision")
                if isinstance(dec, str) and dec.lower() in ("yes", "no"):
                    return dec.lower()
            except Exception:
                # Ignore partial edits or invalid JSON while user is modifying file
                pass

        # Timeout
        if asyncio.get_event_loop().time() - start_time > timeout:
            return "timeout"

        await asyncio.sleep(POLL_INTERVAL)


@function_tool
async def human_approval(proposed_trade: ProposedTrade) -> str:
    """
    Writes the proposed_trade to ./pending_trade.json
    and waits for human approval.

    Returns one of:
        - "yes"
        - "no"
        - "timeout"
    """

    payload = {
        "proposed_trade": proposed_trade.dict(),   # strict structured dict
        "decision": None
    }

    PENDING_FILE.write_text(json.dumps(payload, indent=2), encoding="utf-8")

    print("\n=== HUMAN APPROVAL REQUIRED ===")
    print(f"Wrote proposed trade to: {PENDING_FILE.resolve()}")

    print(
        "\n➡️  Please open the file and set:\n"
        '    {"decision": "yes"}\n'
        "or\n"
        '    {"decision": "no"}\n'
        "Then save the file.\n"
    )

    decision = await _wait_for_decision()
    print("Human decision:", decision)

    return decision


In [43]:
polygon_api_key = os.getenv("POLYGON_API_KEY")

In [44]:

market_mcp = ({"command": "uv", "args": ["run", "market_server.py"]})

trader_mcp_server_params = [
    {"command": "uv", "args": ["run", "accounts_server.py"]},
    {"command": "uv", "args": ["run", "push_server.py"]},
    market_mcp
]

In [70]:
researcher_mcp_server_params = [
    {"command": "uvx", "args": ["mcp-server-fetch"]},
    {"command": "uv", "args": ["run", "search_server.py"]}
]

In [71]:
researcher_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=90) for params in researcher_mcp_server_params]
trader_mcp_servers = [MCPServerStdio(params, client_session_timeout_seconds=30) for params in trader_mcp_server_params]
mcp_servers = trader_mcp_servers + researcher_mcp_servers

In [72]:
async def get_researcher(mcp_servers) -> Agent:
    instructions = f"""You are a financial researcher. You are able to search the web for interesting financial news,
look for possible trading opportunities, and help with research.
Based on the request, you carry out necessary research and respond with your findings.
Take time to make multiple searches to get a comprehensive overview, and then summarize your findings.
If there isn't a specific request, then just respond with investment opportunities based on searching latest news.
The current datetime is {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}
"""
    researcher = Agent(
        name="Researcher",
        instructions=instructions,
        model=gemini_model,
        mcp_servers=mcp_servers,
    )
    return researcher

In [73]:
async def get_researcher_tool(mcp_servers) -> Tool:
    researcher = await get_researcher(mcp_servers)
    return researcher.as_tool(
            tool_name="Researcher",
            tool_description="This tool researches online for news and opportunities, \
                either based on your specific request to look into a certain stock, \
                or generally for notable financial news and opportunities. \
                Describe what kind of research you're looking for."
        )

In [74]:
research_question = "What's the latest news on Disney Stock?"

for server in researcher_mcp_servers:
    await server.connect()
researcher = await get_researcher(researcher_mcp_servers)
with trace("Researcher"):
    result = await Runner.run(researcher, research_question, max_turns=30)
display(Markdown(result.final_output))



Based on the search results, here's a summary of the latest news regarding Disney (DIS) stock:

*   **Stock Performance:** Disney shares experienced a drop after a recent earnings report, with investors seemingly focusing on challenges and trends.
*   **Key Stats:** Recent data indicates the stock's price movements, including open, high, low, previous close, and 52-week high/low ranges.
*   **Earnings Issues:** Disney's earnings report had some significant problems, contributing to the stock's decline. High Hollywood movie budgets are also a concern.

In [75]:
ken_initial_strategy = "You are a day trader that aggressively buys and sells shares based on news and market conditions."
Account.get("ken").reset(ken_initial_strategy)

display(Markdown(await read_accounts_resource("ken")))
display(Markdown(await read_strategy_resource("ken")))

{"name": "ken", "balance": 10000.0, "strategy": "You are a day trader that aggressively buys and sells shares based on news and market conditions.", "holdings": {}, "transactions": [], "portfolio_value_time_series": [["2025-11-17 04:32:04", 10000.0]], "total_portfolio_value": 10000.0, "total_profit_loss": 0.0}

You are a day trader that aggressively buys and sells shares based on news and market conditions.

In [76]:
agent_name = "ken"

# Using MCP Servers to read resources
account_details = await read_accounts_resource(agent_name)
strategy = await read_strategy_resource(agent_name)


In [81]:
instructions = f"""
You are a trader who manages a portfolio for {agent_name}.
You have access to tools that allow you to research news, check stock prices, and buy and sell shares.

---------------------
OVERALL BEHAVIOR RULES
---------------------
• In a single run of this agent, you may make **at most 3 trade proposals**.
• After each approved trade, continue researching and monitoring the market.
• You may propose fewer than 3 trades if the market does not warrant more.
• You should perform research continuously, even between trades.
• You MUST request human approval before executing each trade.
• If human approval is rejected, rethink the proposal and present an alternative (still respecting the 3-trade limit).
• Never request confirmation outside the human_approval tool.
• Never pause or wait for the user except when calling the human_approval tool.

---------------------
WORKFLOW LOOP
---------------------
For each trading cycle (up to 3 cycles):

1. Research:
   - Search the market using your tools.
   - Review relevant news, price changes, sentiment, and fundamentals.
   - Evaluate whether a trade is warranted.

2. If NO trade is warranted:
   - Provide a brief explanation of why you are not making a proposal.
   - Continue researching (but still count this as 0 toward the 3-trade limit).

3. If a trade IS warranted:
   a. Clearly explain your reasoning.
   b. Provide detailed analysis including:
      - market conditions
      - latest news
      - price movements
      - risks
      - expected outcomes
      - alignment with the strategy
   c. Construct a JSON trade proposal object with EXACTLY these fields:
      {{
        "action": "...",               # buy or sell
        "reasoning": "...",            
        "risks": "...",
        "expected_outcome": "...",
        "symbol": "...",               
        "quantity": ...                
      }}

   d. Call:
      human_approval(proposed_trade=<JSON object above>)

   e. If human approves ("yes"), execute the trade using your tools.

   f. If human rejects ("no"):
      - Rethink the trade.
      - Propose an alternative (this still counts toward the same trade attempt).
      - Call human_approval again for the revised proposal.

4. After a trade is completed or rejected:
   - Continue researching.
   - Move on to the next cycle until you reach **3 total trade proposals**.

---------------------
TOOLS YOU MAY USE
---------------------
• Web search tools to gather financial news and sentiment.
• Price-checking tools to inspect stock data.
• Trade execution tools (buy/sell).
• Memory tools to store important context.
• The human_approval tool for mandatory human sign-off.

---------------------
ACCOUNT CONTEXT
---------------------
Your strategy:
{strategy}

Your current account:
{account_details}

Begin the trading session now and manage the portfolio responsibly.
After completing all the trades, provide a summary of all the actions that you took and use the push tool to send a summary notification.
"""


In [82]:
prompt = """
Use your tools to make decisions about your portfolio.
Investigate the news and the market, make your decision, make the trades, and respond with a summary of your actions.
"""

In [83]:
for server in mcp_servers:
    await server.connect()

researcher_tool = await get_researcher_tool(researcher_mcp_servers)
trader = Agent(
    name=agent_name,
    instructions=instructions,
    tools=[researcher_tool, human_approval],
    mcp_servers=trader_mcp_servers,
    model=gemini_model,
)
with trace(agent_name):
    result = await Runner.run(trader, prompt, max_turns=30)
display(Markdown(result.final_output))

/var/folders/00/cm5ct_vx7h34nbv3n33dhqf00000gn/T/ipykernel_64308/4167156196.py:61: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  "proposed_trade": proposed_trade.dict(),   # strict structured dict



=== HUMAN APPROVAL REQUIRED ===
Wrote proposed trade to: /Users/srijanshukla/projects/agents/6_mcp/pending_trade.json

➡️  Please open the file and set:
    {"decision": "yes"}
or
    {"decision": "no"}
Then save the file.

Human decision: yes

=== HUMAN APPROVAL REQUIRED ===
Wrote proposed trade to: /Users/srijanshukla/projects/agents/6_mcp/pending_trade.json

➡️  Please open the file and set:
    {"decision": "yes"}
or
    {"decision": "no"}
Then save the file.

Human decision: yes

=== HUMAN APPROVAL REQUIRED ===
Wrote proposed trade to: /Users/srijanshukla/projects/agents/6_mcp/pending_trade.json

➡️  Please open the file and set:
    {"decision": "yes"}
or
    {"decision": "no"}
Then save the file.

Human decision: yes


Okay, I have completed the trading session and sent a summary notification.


In [84]:
import json
from IPython.display import HTML

raw = json.loads(await read_accounts_resource(agent_name))
formatted = json.dumps(raw, indent=4)

HTML(f"""
<div style='max-height: 500px; overflow-y: scroll; border: 1px solid #ccc; padding: 10px;'>
<pre>{formatted}</pre>
</div>
""")
